
# AI-Assisted RTL Reconstruction

**CSAW 2026 AI Hardware Attack Challenge — The Bitstream Bandits**

This notebook uses an LLM to reconstruct readable SystemVerilog from a recovered Lattice iCE40 FPGA netlist. The recovered design implements a cryptographic accelerator with an SPI interface.

The initial reconstruction focuses on the 32-bit SPI shift register and its registered MISO output. The cryptographic datapath and controller are left unchanged.

## 1. Environment Setup

Initialize project paths, the Portkey API client, and the model configuration. Generated RTL, verification artifacts, and AI interaction logs are saved separately.


In [2]:
from pathlib import Path
from collections import deque
from dotenv import load_dotenv
from portkey_ai import Portkey
import os, re, subprocess

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

load_dotenv(ROOT / ".env", override=True)

SOURCE = ROOT / "recovery/recovered_pins.v"
OUTPUT = ROOT / "rtl/readable/readable_spi_shift.sv"
BUILD = ROOT / "build"
LOG_DIR = ROOT / "ai_logs/readable_rtl"

for path in (OUTPUT.parent, BUILD, LOG_DIR):
    path.mkdir(parents=True, exist_ok=True)

MODEL = os.getenv("PORTKEY_MODEL", "@vertexai/anthropic.claude-opus-5-5")

client = Portkey(
    api_key=os.environ["PORTKEY_API_KEY"],
    base_url=os.getenv(
        "PORTKEY_BASE_URL",
        "https://ai-gateway.apps.cloud.rt.nyu.edu/v1"
    ),
    max_retries=0
)

# False: reuse existing generated RTL without another API call.
# True: ask Claude to generate a new candidate.
REGENERATE = False

print("Project:", ROOT)
print("Model:", MODEL)

Project: /home/hansyang/projects/aha2026/aha2026-course
Model: @vertexai/anthropic.claude-opus-5-5



## 2. SPI Shift Register Recovery

Analyze the recovered structural Verilog to identify state registers, combinational dependencies, and the serial data path.

Starting from the register driving MISO, trace backward through connected registers to the MOSI input. Extract the original register updates and LUT equations as evidence for the LLM.

The recovered SPI interface contains 32 shift-register stages followed by a separate registered MISO output. External control and parallel-load signals are retained as inputs.


In [3]:
source = SOURCE.read_text()
lines = source.splitlines()

signals = lambda expr: set(
    re.findall(r"\b(?:n\d+|pin_\d+)\b", expr)
)

registers = set()
initial_values = {}

for match in re.finditer(r"\breg\s+([^;]+);", source, re.S):
    for name, value in re.findall(
        r"\b(n\d+|pin_\d+)\s*=\s*([01])\b",
        match.group(1)
    ):
        registers.add(name)
        initial_values[name] = int(value)

assignments = {}
updates = {}

for i, line in enumerate(lines, 1):
    m = re.match(r"\s*assign\s+(n\d+|pin_\d+)\s*=\s*(.*);", line)
    if m:
        assignments[m.group(1)] = (i, m.group(2))

    m = re.search(r"\b(n\d+|pin_\d+)\s*<=\s*(.*);", line)
    if m and m.group(1) in registers:
        updates[m.group(1)] = (i, m.group(2))

def fanin(reg):
    """Trace combinational logic until reaching other registers or pins."""
    queue = deque(signals(updates[reg][1]))
    parents, boundary, logic = set(), set(), {}
    visited = set()

    while queue:
        sig = queue.popleft()
        if sig in visited:
            continue
        visited.add(sig)

        if sig in registers:
            parents.add(sig)
        elif sig in assignments:
            logic[sig] = assignments[sig]
            queue.extend(signals(assignments[sig][1]))
        else:
            boundary.add(sig)

    return parents, boundary, logic

# Follow the serial shift chain backward.
chain = []
current = "n800"

while current not in chain:
    chain.append(current)
    parents, _, _ = fanin(current)

    if not parents:
        break
    if len(parents) != 1:
        raise RuntimeError(f"Ambiguous predecessor of {current}: {parents}")

    current = next(iter(parents))

chain.reverse()

assert len(chain) == 32
assert chain[0] == "n628" and chain[-1] == "n800"

state = chain + ["pin_31"]
logic = {}
boundary = {"n1", "n2", "pin_36"}

for reg in state:
    parents, external, equations = fanin(reg)
    logic.update(equations)
    boundary.update(external)
    boundary.update(parents - set(state))

boundary.difference_update(state)

# Original combinational equations and sequential statements.
evidence = ["SHIFT ORDER: " + " -> ".join(state)]

for reg in state:
    number, _ = updates[reg]
    evidence.append(f"\nREGISTER {reg}, initial={initial_values[reg]}")
    evidence.extend(lines[max(0, number - 3):number])

for sig, (number, _) in sorted(logic.items(), key=lambda x: x[1][0]):
    evidence.append(f"{number}: {lines[number - 1]}")

evidence_text = "\n".join(evidence)
(LOG_DIR / "spi_evidence.txt").write_text(evidence_text)

print("State registers:", len(registers))
print("SPI stages:", len(chain))
print("Combinational equations:", len(logic))
print("External signals:", len(boundary))
print("Evidence tokens (approx.):", len(evidence_text) // 3)

State registers: 71
SPI stages: 32
Combinational equations: 33
External signals: 37
Evidence tokens (approx.): 2641



## 3. AI-Assisted Architecture Analysis

Use Claude Opus 5.5 to interpret the recovered netlist and identify functional components of the FPGA accelerator.

The model examines the extracted SPI register dependencies, documented interface, and recovered logic to distinguish confirmed structural relationships from architectural hypotheses.

This analysis provides context for subsequent RTL reconstruction without assuming the cryptographic algorithm or unsupported internal behavior.


In [8]:
spec_path = ROOT / "docs/transaction_specification.md"
spec = spec_path.read_text() if spec_path.exists() else ""

architecture_prompt = f"""
Analyze the architecture of this recovered Lattice iCE40
cryptographic accelerator.

Documented interface:
{spec}

Recovered SPI register chain:
{chain}

External boundary signals:
{sorted(boundary)}

Extracted structural RTL:
{evidence_text}

Identify:
1. SPI interface organization and shift-register behavior.
2. Clocking, enable, and reset behavior.
3. Relationship between SPI and cryptographic datapath.
4. What can be determined about encryption/decryption.
5. What remains unknown.

Separate confirmed facts from hypotheses.
Do not invent an encryption algorithm or register functions.
Keep the response under 800 words.
"""

architecture_log = LOG_DIR / "architecture"

# Reuse existing analysis unless explicitly regenerated.
if REGENERATE or not architecture_log.with_suffix(".txt").exists():
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an FPGA reverse-engineering engineer. "
                    "Ground conclusions in provided structural evidence."
                )
            },
            {"role": "user", "content": architecture_prompt}
        ],
        max_tokens=8192,
        timeout=300
    )

    analysis = response.choices[0].message.content or ""

    architecture_log.with_suffix(".txt").write_text(
        analysis, encoding="utf-8"
    )

    architecture_log.with_name("architecture_prompt.txt").write_text(
        architecture_prompt, encoding="utf-8"
    )

    architecture_log.with_name("architecture_response.json").write_text(
        response.model_dump_json(indent=2), encoding="utf-8"
    )
else:
    analysis = architecture_log.with_suffix(".txt").read_text()

print(analysis)

# Recovered iCE40 Crypto Accelerator: Architecture Analysis

## 1. SPI interface and shift register

**Confirmed from the netlist**

- **Chain size and clock.** There is one 32-flop chain, `n628 → … → n800`, plus a 33rd output flop `pin_31`. All of them clock on `pin_36`.
- **Same next-state form in every stage.** I reduced each LUT to one expression:
  - `next = pin_28 ? !ext_i : prev_stage`
  - Some LUTs are written as nested muxes, for example `n1515`, `n1707` and `n1787`. Expanding them gives exactly this form. For instance, `n1515` reduces to `pin_28 ? !n1032 : n1024`.
- **Serial path.** Stage 0 is `n2169 = pin_28 ? !n630 : pin_32`. So `pin_32` is the serial input, and `pin_31 <= n800` is the serial output.
- **Bit order.** The first bit received travels to `n800` after 32 enabled shifts. This gives FIFO, MSB-first behaviour, consistent with the documented SPI mode.
- **Parallel load.** When `pin_28=1`, every stage loads the inverse of one of exactly 32 external nets. Stage order 


## 4. AI-Assisted RTL Generation

Provide the extracted register chain, original logic equations, and interface mappings to Claude Opus 5.5.

The model is instructed to produce a readable, synthesizable SystemVerilog implementation while preserving register ordering, initialization, clocking, reset and enable behavior, and parallel-load functionality.

The generated RTL is saved to `rtl/readable/readable_spi_shift.sv`. Prompts, responses, and model metadata are recorded for reproducibility. Existing RTL can be reused without repeating the API call.


In [4]:
SYSTEM_PROMPT = """
You are an expert FPGA and SystemVerilog engineer.
Convert recovered structural Verilog into readable synthesizable RTL.

Preserve exact register ordering, initialization, clocking,
enable/reset behavior, parallel-load equations, and MISO timing.
Do not guess the cryptographic algorithm or omit any state.
Return one complete SystemVerilog code block only.
"""

USER_PROMPT = f"""
Generate module readable_spi_shift.

Pin mappings:
pin_36 = clk
pin_28 = cs_n
pin_32 = mosi
pin_31 = miso
n1 = enable
n2 = reset

Declare every other external signal as a one-bit input:
{sorted(boundary)}

The module contains 32 shift stages and one additional
registered MISO output.

Maintain exact behavior for arbitrary boundary inputs.

ORIGINAL RECOVERED LOGIC:
{evidence_text}
"""

if REGENERATE or not OUTPUT.exists():
    generation_dir = LOG_DIR / "generation"
    generation_dir.mkdir(exist_ok=True)

    attempt = 1
    while (generation_dir / f"attempt_{attempt:03d}").exists():
        attempt += 1

    run_dir = generation_dir / f"attempt_{attempt:03d}"
    run_dir.mkdir()

    (run_dir / "prompt.txt").write_text(
        f"Model: {MODEL}\n\n{SYSTEM_PROMPT}\n\n{USER_PROMPT}"
    )

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT}
        ],
        max_tokens=8192,
        timeout=300
    )

    result = response.choices[0].message.content or ""

    (run_dir / "response.txt").write_text(result)
    (run_dir / "response_raw.json").write_text(
        response.model_dump_json(indent=2)
    )

    if response.choices[0].finish_reason not in ("stop", "end_turn"):
        raise RuntimeError("Incomplete AI generation")

    match = re.search(
        r"```(?:systemverilog|verilog|sv)\s*\n(.*?)```",
        result, re.S | re.I
    )

    if not match:
        raise ValueError("Claude returned no SystemVerilog code block")

    OUTPUT.write_text(match.group(1).strip() + "\n")
    (run_dir / OUTPUT.name).write_text(OUTPUT.read_text())

    print("Generated with:", response.model)
    print("Tokens:", response.usage.total_tokens)
else:
    print("Reusing existing generated RTL")

print("RTL:", OUTPUT.relative_to(ROOT))

Reusing existing generated RTL
RTL: rtl/readable/readable_spi_shift.sv



## 5. Behavioral Equivalence Verification

Construct an isolated reference model using the original recovered Verilog equations and compare it against the AI-generated implementation.

Both modules receive identical boundary inputs over 2,000 simulated clock cycles. The test includes serial shifting, parallel loading, and randomized control signals.

The test checks the registered MISO output after each clock edge. Compilation and simulation results are recorded automatically.

**Limitation:** Passing this test demonstrates behavioral agreement for the tested input sequences, not formal equivalence. Internal register states and integration with the full cryptographic accelerator are not independently verified.


In [5]:
# Original reference module.
ports = [f"input wire {sig}" for sig in sorted(boundary)]
ports.append(f"output reg pin_31 = 1'b{initial_values['pin_31']}")

ref = ["module reference_spi_shift (" + ", ".join(ports) + ");"]

for reg in chain:
    ref.append(f"reg {reg} = 1'b{initial_values[reg]};")

for sig in sorted(logic):
    ref.append(f"wire {sig};")

for sig, (number, _) in sorted(logic.items(), key=lambda x: x[1][0]):
    ref.append(lines[number - 1])

for reg in state:
    number, _ = updates[reg]
    preceding = "\n".join(lines[max(0, number - 5):number])

    if "always @(posedge pin_36)" not in preceding:
        raise RuntimeError(f"Unexpected clock for {reg}")

    ref.append(
        "always @(posedge pin_36) begin\n"
        + lines[number - 1].strip()
        + "\nend"
    )

ref.append("endmodule")

ref_file = BUILD / "reference_spi_shift.sv"
ref_file.write_text("\n".join(ref) + "\n")

# Map original pin names to readable RTL ports.
names = {
    "pin_36": "clk",
    "pin_28": "cs_n",
    "pin_32": "mosi",
    "n1": "enable",
    "n2": "reset"
}
mapped = lambda sig: names.get(sig, sig)

inputs = sorted(boundary)
declarations = "\n".join(f"reg {mapped(s)} = 0;" for s in inputs)

ref_ports = ",\n".join(
    f".{s}({mapped(s)})" for s in inputs
)
ai_ports = ",\n".join(
    f".{mapped(s)}({mapped(s)})" for s in inputs
)

randomize = "\n".join(
    f"{mapped(s)} = $random;"
    for s in inputs if s != "pin_36"
)

tb = f"""
`timescale 1ns/1ps
module test;

{declarations}
wire expected, actual;

reference_spi_shift reference_dut (
{ref_ports},
.pin_31(expected)
);

readable_spi_shift ai_dut (
{ai_ports},
.miso(actual)
);

integer cycle;

initial begin
    #1;
    if (expected !== actual)
        $fatal(1, "Initial output mismatch");

    for (cycle = 0; cycle < 2000; cycle = cycle + 1) begin
        clk = 0;

        {randomize}

        if (cycle < 500 || (cycle >= 1000 && cycle < 1500)) begin
            cs_n = 0;
            enable = 1;
            reset = 0;
        end
        else if (cycle < 1000) begin
            cs_n = 1;
            enable = 1;
            reset = 0;
        end

        #5 clk = 1;
        #1;

        if (expected !== actual)
            $fatal(1, "Mismatch at cycle %0d", cycle);

        #4 clk = 0;
    end

    $display("PASS: SPI MISO equivalence (2000 cycles)");
    $finish;
end
endmodule
"""

tb_file = BUILD / "test_spi_shift.sv"
tb_file.write_text(tb)

executable = BUILD / "test_spi_shift.vvp"

compile_result = subprocess.run(
    [
        "iverilog", "-g2012", "-s", "test",
        "-o", str(executable),
        str(ref_file), str(OUTPUT), str(tb_file)
    ],
    capture_output=True, text=True
)

if compile_result.returncode:
    raise RuntimeError(compile_result.stderr)

simulation = subprocess.run(
    ["vvp", str(executable)],
    capture_output=True, text=True
)

verification_log = LOG_DIR / "verification.log"
verification_log.write_text(
    compile_result.stdout + compile_result.stderr
    + simulation.stdout + simulation.stderr
)

print(simulation.stdout)

if simulation.returncode:
    raise RuntimeError(simulation.stderr)

verified = True

PASS: SPI MISO equivalence (2000 cycles)
/home/hansyang/projects/aha2026/aha2026-course/build/test_spi_shift.sv:194: $finish called at 20001000 (1ps)




## 6. Results and Summary

Summarize the recovered SPI architecture, generated RTL, compilation status, and behavioral verification results.

The reconstruction produces a readable implementation of the 32-bit SPI shift register and its additional MISO output register. Verification compares the generated implementation against the recovered reference over 2,000 clock cycles.

This demonstrates an AI-assisted workflow for translating structural FPGA logic into readable RTL while preserving observable behavior.

**Next steps:** Analyze the remaining cryptographic datapath and control logic, integrate the reconstructed SPI module, and validate the complete accelerator using the existing encryption/decryption testbench.


In [6]:
print("AI-Assisted RTL Reconstruction")
print("-" * 40)
print("Model:", MODEL)
print("Source:", SOURCE.relative_to(ROOT))
print("Generated RTL:", OUTPUT.relative_to(ROOT))
print("SPI shift stages:", len(chain))
print("MISO output registers:", 1)
print("Combinational equations:", len(logic))
print("Test cycles:", 2000)
print("Compilation: PASS")
print("MISO equivalence:", "PASS" if verified else "FAIL")

print("\nSaved files:")
print("-", OUTPUT.relative_to(ROOT))
print("-", (LOG_DIR / "spi_evidence.txt").relative_to(ROOT))
print("-", verification_log.relative_to(ROOT))

AI-Assisted RTL Reconstruction
----------------------------------------
Model: @vertexai/anthropic.claude-opus-5-5
Source: recovery/recovered_pins.v
Generated RTL: rtl/readable/readable_spi_shift.sv
SPI shift stages: 32
MISO output registers: 1
Combinational equations: 33
Test cycles: 2000
Compilation: PASS
MISO equivalence: PASS

Saved files:
- rtl/readable/readable_spi_shift.sv
- ai_logs/readable_rtl/spi_evidence.txt
- ai_logs/readable_rtl/verification.log
